![image](https://raw.githubusercontent.com/IBM/watson-machine-learning-samples/master/cloud/notebooks/headers/watsonx-Prompt_Lab-Notebook.png)
# Use AutoAI RAG with watsonx Text Extraction service

#### Disclaimers

- Use only Projects and Spaces that are available in the watsonx context.


## Notebook content

This notebook demonstrates how to process data using the IBM watsonx.ai Text Extraction service and use the result in an AutoAI RAG experiment.
The data used in this notebook is from the [Granite Code Models paper](https://arxiv.org/pdf/2405.04324).

Some familiarity with Python is helpful. This notebook uses Python 3.12.


## Learning goal

The learning goals of this notebook are:

- Process data using the IBM watsonx.ai Text Extraction service
- Create an AutoAI RAG job that will find the best RAG pattern based on processed data


## Contents

This notebook contains the following parts:

1. [Set up the environment](#Set-up-the-environment)
2. [Prepare data and connections for the Text Extraction service](#Prepare-data-and-connections-for-the-Text-Extraction-service)
3. [Process data using the Text Extraction service](#Process-data-using-the-Text-Extraction-service)
4. [Prepare data and connections for the AutoAI RAG experiment](#Prepare-data-and-connections-for-the-AutoAI-RAG-experiment)
5. [Run the AutoAI RAG experiment](#Run-the-AutoAI-RAG-experiment)
6. [Compare and test of RAG Patterns](#Compare-and-test-of-RAG-Patterns)
7. [Summary and next steps](#Summary-and-next-steps)

<a id="Set-up-the-environment"></a>
## Set up the environment

Before you use the sample code in this notebook, you must perform the following setup tasks:

-  Contact your IBM Cloud Pak® for Data administrator and ask them for your account credentials

### Install and import the required modules and dependencies

In [1]:
%pip install -U 'ibm-watsonx-ai[rag]>=1.4.10' | tail -n 1
%pip install 'wget'


Note: you may need to restart the kernel to use updated packages.
  Using cached wget-3.2-py3-none-any.whl
Note: you may need to restart the kernel to use updated packages.


### Connect to WML
Authenticate the Watson Machine Learning service on IBM Cloud Pak® for Data. You need to provide the platform `url`, your `username`, and your `api_key`.

- `url` - url which points to your CPD instance.
- `username` - username to your CPD instance.

In [ ]:
url = "PASTE YOUR CPD INSTANCE URL HERE"
username = "PASTE YOUR CPD INSTANCE USERNAME HERE"

In [ ]:
import getpass

from ibm_watsonx_ai import Credentials

credentials = Credentials(
    username=username,
    api_key=getpass.getpass("Enter your watsonx.ai API key and hit enter: "),
    url=url,
    instance_id="openshift",
    version="5.3",
)

Alternatively, you can use your username and password to authenticate WML services.

In [ ]:
if "credentials" not in locals() or not credentials.api_key:
    credentials = Credentials(
        username=username,
        password=getpass.getpass("Enter your watsonx.ai password and hit enter: "),
        url=url,
        instance_id="openshift",
        version="5.3",
    )

### Working with spaces

First, you need to create a space for your work. If you do not have a space already created, you can use `{PLATFORM_URL}/ml-runtime/spaces?context=icp4data` to create one.

- Click **New Deployment Space**
- Create an empty space
- Go to the space `Settings` tab
- Copy `Space GUID` into your env file or else enter it in the window which will show up after running below cell

**Tip**: You can also use SDK to prepare the space for your work. Find more information in the [Space Management sample notebook](https://github.com/IBM/watson-machine-learning-samples/blob/master/cpd5.0/notebooks/python_sdk/instance-management/Space%20management.ipynb).

**Action**: Assign the space ID below

In [ ]:
import os

try:
    SPACE_ID = os.environ["SPACE_ID"]
except KeyError:
    SPACE_ID = input("Please enter your space_id (hit enter): ")

Create an instance of APIClient with authentication details

In [ ]:
from ibm_watsonx_ai import APIClient

client = APIClient(credentials=credentials, space_id=SPACE_ID)

### Create an instance of COS client

Connect to the default COS instance for the provided space by using the `ibm_boto3` package.

In [7]:
import ibm_boto3

cos_credentials = client.spaces.get_details(space_id=SPACE_ID)["entity"]["storage"][
    "properties"
]

cos_client = ibm_boto3.client(
    service_name="s3",
    endpoint_url=cos_credentials["endpoint_url"],
    aws_access_key_id=cos_credentials["credentials"]["editor"]["access_key_id"],
    aws_secret_access_key=cos_credentials["credentials"]["editor"]["secret_access_key"],
)

Create a new bucket.

In [8]:
cos_bucket_name = "autoai-rag-with-extraction-experiment"

buckets_names = [bucket["Name"] for bucket in cos_client.list_buckets()["Buckets"]]
if not cos_bucket_name in buckets_names:
    cos_client.create_bucket(Bucket=cos_bucket_name)

Initialize the client connection to the created bucket and get the connection ID.

In [9]:
connection_details = client.connections.create(
    {
        "datasource_type": client.connections.get_datasource_type_uid_by_name(
            "bluemixcloudobjectstorage"
        ),
        "name": "Connection to COS for tests",
        "properties": {
            "bucket": cos_bucket_name,
            "access_key": cos_credentials["credentials"]["editor"]["access_key_id"],
            "secret_key": cos_credentials["credentials"]["editor"]["secret_access_key"],
            "iam_url": client.service_instance._href_definitions.get_iam_token_url(),
            "url": cos_credentials["endpoint_url"],
        },
    }
)

cos_connection_id = client.connections.get_id(connection_details)

Creating connections...
SUCCESS


<a id="Prepare-data-and-connections-for-the-Text-Extraction-service"></a>
## Prepare data and connections for the Text Extraction service

The document, from which we are going to extract text, is located in the IBM Cloud Object Storage (COS). In this notebook, we will use the [Granite Code Models paper](https://arxiv.org/pdf/2405.04324) as a source text document. The final results file, which will contain extracted text and necessary metadata, will be placed in the COS. So we will use the `ibm_watsonx_ai.helpers.DataConnection` and the `ibm_watsonx_ai.helpers.S3Location` class to create Python objects that will represent the references to the processed files. Reference to the final results will be used as an input for the AutoAI RAG experiment. 

In [10]:
from ibm_watsonx_ai.helpers import DataConnection, S3Location

data_url = "https://arxiv.org/pdf/2405.04324"

te_input_filename = "granite_code_models_paper.pdf"
te_result_filename = "granite_code_models_paper.md"

Download and upload training data to the COS bucket. Then define a connection to the uploaded file.

In [11]:
import wget

wget.download(data_url, te_input_filename)
cos_client.upload_file(te_input_filename, cos_bucket_name, te_input_filename)

Input file connection.

In [12]:
input_data_reference = DataConnection(
    connection_asset_id=cos_connection_id,
    location=S3Location(bucket=cos_bucket_name, path=te_input_filename),
)
input_data_reference.set_client(client)

Output file connection.

In [13]:
result_data_reference = DataConnection(
    connection_asset_id=cos_connection_id,
    location=S3Location(bucket=cos_bucket_name, path=te_result_filename),
)
result_data_reference.set_client(client)

<a id="Process-data-using-the-Text-Extraction-service"></a>
## Process data using the Text Extraction service

Initialize the Text Extraction service endpoint.

In [14]:
from ibm_watsonx_ai.foundation_models.extractions import TextExtractionsV2

extraction = TextExtractionsV2(
    credentials=credentials,
    space_id=SPACE_ID,
)

Run a text extraction job for connections created in the previous step.

In [15]:
from ibm_watsonx_ai.foundation_models.extractions import TextExtractionsV2ResultFormats
from ibm_watsonx_ai.metanames import TextExtractionsMetaNames

response = extraction.run_job(
    document_reference=input_data_reference,
    results_reference=result_data_reference,
    parameters={
        TextExtractionsMetaNames.OCR: {
            "process_image": True,
            "languages_list": ["en"],
        },
        TextExtractionsMetaNames.TABLE_PROCESSING: {"enabled": True},
    },
    result_formats=[TextExtractionsV2ResultFormats.MARKDOWN],
)

job_id = response["metadata"]["id"]

Get the text extraction result.

In [16]:
from IPython.display import Markdown, display

cos_client.download_file(
    Bucket=cos_bucket_name, Key=te_result_filename, Filename=te_result_filename
)

with open(te_result_filename, "r", encoding="utf-8") as file:
    # Display beginning of the result file
    display(Markdown((file.read()[:3000])))

## Granite Code Models: A Family of Open Foundation Models for Code Intelligence

Mayank Mishra⋆ Matt Stallone⋆ Gaoyuan Zhang⋆ Yikang Shen Aditya Prasad Adriana Meza Soria Michele Merler Parameswaran Selvam Saptha Surendran Shivdeep Singh Manish Sethi Xuan-Hong Dang Pengyuan Li Kun-Lung Wu Syed Zawad Andrew Coleman Matthew White Mark Lewis Raju Pavuluri Yan Koyfman Boris Lublinsky Maximilien de Bayser Ibrahim Abdelaziz Kinjal Basu Mayank Agarwal Yi Zhou Chris Johnson Aanchal Goyal Hima Patel Yousaf Shah Petros Zerfos Heiko Ludwig Asim Munawar Maxwell Crouse Pavan Kapanipathi Shweta Salaria Bob Calio Sophia Wen Seetharami Seelam Brian Belgodere Carlos Fonseca Amith Singhee Nirmit Desai David D. Cox Ruchir Puri† Rameswar Panda†

IBM Research ⋆Equal Contribution

†Corresponding Authors ruchir@us.ibm.com, rpanda@ibm.com

## Abstract

Large Language Models (LLMs) trained on code are revolutionizing the software development process. Increasingly, code LLMs are being inte grated into software development environments to improve the produc tivity of human programmers, and LLM-based agents are beginning to show promise for handling complex tasks autonomously. Realizing the full potential of code LLMs requires a wide range of capabilities, including code generation, fixing bugs, explaining and documenting code, maintaining repositories, and more. In this work, we introduce the Granite series of decoder-only code models for code generative tasks, trained with code written in 116 programming languages. The Granite Code models family consists of models ranging in size from 3 to 34 billion parameters, suitable for applications ranging from complex application modernization tasks to on-device memory-constrained use cases. Evaluation on a comprehensive set of tasks demonstrates that Granite Code models consistently reaches state-of-the-art performance among available open-source code LLMs. The Granite Code model family was optimized for enterprise software devel opment workflows and performs well across a range of coding tasks (e.g. code generation, fixing and explanation), making it a versatile “all around” code model. We release all our Granite Code models under an Apache 2.0 license for both research and commercial use.

 https://github.com/ibm-granite/granite-code-models

## 1 Introduction

Over the last several decades, software has been woven into the fabric of every aspect of our society. As demand for software development surges, it is more critical than ever to increase software development productivity, and LLMs provide promising path for augmenting human programmers. Prominent enterprise use cases for LLMs in software development productivity include code generation, code explanation, code fixing, unit test and documentation generation, application modernization, vulnerability detection, code translation, and more.

Recent years have seen rapid progress in LLM’s ability to generate and manipulate code, and a range of models with impressive coding abi

<a id="Prepare-data-and-connections-for-the-AutoAI-RAG-experiment"></a>
## Prepare data and connections for the AutoAI RAG experiment

Upload a `json` file to use for benchmarking to COS and define a connection to this file. 

Note: `correct_answer_document_ids` must refer to the document processed by text extraction service, not the initial document.

In [17]:
benchmarking_data = [
    {
        "question": "What are the two main variants of Granite Code models?",
        "correct_answer": "The two main variants are Granite Code Base and Granite Code Instruct.",
        "correct_answer_document_ids": [te_result_filename],
    },
    {
        "question": "What is the purpose of Granite Code Instruct models?",
        "correct_answer": "Granite Code Instruct models are finetuned for instruction-following tasks using datasets like CommitPack, OASST, HelpSteer, and synthetic code instruction datasets, aiming to improve reasoning and instruction-following capabilities.",
        "correct_answer_document_ids": [te_result_filename],
    },
    {
        "question": "What is the licensing model for Granite Code models?",
        "correct_answer": "Granite Code models are released under the Apache 2.0 license, ensuring permissive and enterprise-friendly usage.",
        "correct_answer_document_ids": [te_result_filename],
    },
]

In [18]:
import os

test_filename = "benchmark.json"

if not os.path.isfile(test_filename):
    with open(test_filename, "w") as json_file:
        json.dump(benchmarking_data, json_file, indent=4)

cos_client.upload_file(test_filename, cos_bucket_name, test_filename)

Test the data connection.

In [19]:
test_data_reference = DataConnection(
    connection_asset_id=cos_connection_id,
    location=S3Location(bucket=cos_bucket_name, path=test_filename),
)
test_data_reference.set_client(client)

test_data_references = [test_data_reference]

Use the reference to the Text Extraction job result as input for the AutoAI RAG experiment.

In [20]:
input_data_references = [result_data_reference]

<a id="Run-the-AutoAI-RAG-experiment"></a>
## Run the AutoAI RAG experiment

Provide the input information for AutoAI RAG optimizer:
- `name` - experiment name
- `description` - experiment description
- `max_number_of_rag_patterns` - maximum number of RAG patterns to create
- `optimization_metrics` - target optimization metrics

In [21]:
from ibm_watsonx_ai.experiment import AutoAI

experiment = AutoAI(credentials, space_id=SPACE_ID)

rag_optimizer = experiment.rag_optimizer(
    name="AutoAI RAG - Text Extraction service experiment",
    description="AutoAI RAG experiment on documents generated by text extraction service",
    max_number_of_rag_patterns=5,
    optimization_metrics=["answer_correctness"],
)

Call the `run()` method to trigger the AutoAI RAG experiment. Choose one of two modes: 

- To use the **interactive mode** (synchronous job), specify `background_mode=False` 
- To use the **background mode** (asynchronous job), specify `background_mode=True`

In [22]:
rag_optimizer.run(
    input_data_references=input_data_references,
    test_data_references=test_data_references,
    background_mode=False,
)



##############################################

Running 'cae26b9b-b959-45e0-a7f8-a272bc0e6aba'

##############################################


pending....
running...................................
completed
Training of 'cae26b9b-b959-45e0-a7f8-a272bc0e6aba' finished successfully.


{'entity': {'hardware_spec': {'id': 'a6c4923b-b8e4-444c-9f43-8a7ec3020110',
   'name': 'L'},
  'input_data_references': [{'connection': {'id': '549e8fd2-da55-4d88-b51e-56bdc6d82d49'},
    'location': {'bucket': 'autoai-rag-with-extraction-experiment',
     'file_name': 'granite_code_models_paper.md'},
    'type': 'connection_asset'}],
  'parameters': {'constraints': {'max_number_of_rag_patterns': 5},
   'optimization': {'metrics': ['answer_correctness']},
   'output_logs': True},
  'results': [{'context': {'iteration': 0,
     'max_combinations': 360,
     'rag_pattern': {'composition_steps': ['model_selection',
       'chunking',
       'embeddings',
       'retrieval',
       'generation'],
      'duration_seconds': 17,
      'location': {'evaluation_results': 'default_autoai_rag_out/cae26b9b-b959-45e0-a7f8-a272bc0e6aba/Pattern1/evaluation_results.json',
       'indexing_notebook': 'default_autoai_rag_out/cae26b9b-b959-45e0-a7f8-a272bc0e6aba/Pattern1/indexing_inference_notebook.ipynb

<a id="Compare-and-test-of-RAG-Patterns"></a>
## Compare and test of RAG Patterns

You can list the trained patterns and information on evaluation metrics in the form of a Pandas DataFrame by calling the `summary()` method. You can use the DataFrame to compare all discovered patterns and select the one you like for further testing.

In [23]:
summary = rag_optimizer.summary()
summary

,mean_answer_correctness,chunking.method,chunking.chunk_size,chunking.chunk_overlap,embeddings.model_id,vector_store.distance_metric,retrieval.method,retrieval.number_of_chunks,generation.model_id,agent.type
Pattern_Name,,,,,,,,,,
Pattern2,0.9039,semantic,1024,0,ibm/slate-125m-english-rtrvr-v2,cosine,window,3,ibm/granite-4-h-small,sequential
Pattern5,0.8915,semantic,1024,0,ibm/slate-125m-english-rtrvr-v2,cosine,window,3,ibm/granite-4-h-small,sequential
Pattern3,0.8051,semantic,1024,0,ibm/slate-125m-english-rtrvr-v2,cosine,window,3,meta-llama/llama-3-3-70b-instruct,sequential
Pattern4,0.7937,recursive,1024,512,intfloat/multilingual-e5-large,cosine,window,3,meta-llama/llama-3-3-70b-instruct,sequential
Pattern1,0.7690,semantic,1024,0,ibm/slate-125m-english-rtrvr-v2,cosine,simple,5,ibm/granite-3-8b-instruct,sequential


### Get the selected pattern

Get the RAGPattern object from the RAG Optimizer experiment. By default, the RAGPattern of the best pattern is returned.

In [24]:
best_pattern_name = summary.index.values[0]
print("Best pattern is:", best_pattern_name)

best_pattern = rag_optimizer.get_pattern()

Best pattern is: Pattern2
  Using cached pyarrow-22.0.0-cp312-cp312-macosx_12_0_arm64.whl.metadata (3.2 kB)
Using cached pyarrow-22.0.0-cp312-cp312-macosx_12_0_arm64.whl (34.2 MB)


In [25]:
rag_optimizer.get_pattern_details(pattern_name=best_pattern_name)

{'composition_steps': ['model_selection',
  'chunking',
  'embeddings',
  'retrieval',
  'generation'],
 'duration_seconds': 10,
 'location': {'evaluation_results': 'default_autoai_rag_out/cae26b9b-b959-45e0-a7f8-a272bc0e6aba/Pattern2/evaluation_results.json',
  'indexing_notebook': 'default_autoai_rag_out/cae26b9b-b959-45e0-a7f8-a272bc0e6aba/Pattern2/indexing_inference_notebook.ipynb',
  'inference_notebook': 'default_autoai_rag_out/cae26b9b-b959-45e0-a7f8-a272bc0e6aba/Pattern2/indexing_inference_notebook.ipynb',
  'inference_service_code': 'default_autoai_rag_out/cae26b9b-b959-45e0-a7f8-a272bc0e6aba/Pattern2/inference_ai_service.gz',
  'inference_service_metadata': 'default_autoai_rag_out/cae26b9b-b959-45e0-a7f8-a272bc0e6aba/Pattern2/inference_service_metadata.json'},
 'name': 'Pattern2',
 'settings': {'agent': {'description': 'Sequential graph with single index retriever.',
   'framework': 'langgraph',
   'type': 'sequential'},
  'chunking': {'chunk_overlap': 0, 'chunk_size': 1024, 

Test the RAGPattern by querying it locally.

In [27]:
from ibm_watsonx_ai.deployments import RuntimeContext

runtime_context = RuntimeContext(api_client=client)
inference_service_function = best_pattern.inference_service(runtime_context)[0]

In [ ]:
question = "Which industry players are mentioned as IBM’s strategic partners?"

context = RuntimeContext(
    api_client=client,
    request_payload_json={"messages": [{"role": "user", "content": question}]},
)

In [29]:
print(inference_service_function(context)["body"]["choices"][0]["message"]["content"])

Based on the provided context, the following industry players are mentioned as IBM's strategic partners:

1. Meta - IBM collaborated with Meta on the Llama 3 model card.

2. Cohere - IBM mentions Cohere's Command R+ model.

3. Databricks - IBM references Databricks' DBRX model.

4. Mistral AI - IBM cites Mistral AI's Mixtral 8x22B model.

5. OpenAssistant - IBM acknowledges OpenAssistant's work on democratizing large language model alignment.

6. StarCoder - IBM mentions StarCoder and StarCoder 2 models.

7. The Stack - IBM refers to the Stack v2 dataset used for training code models.

So in summary, the key industry partners mentioned are Meta, Cohere, Databricks, Mistral AI, OpenAssistant, and the organizations behind the StarCoder and Stack projects.


### Deploy the RAGPattern

Store the defined RAG function and create a deployed asset to deploy the RAGPattern.

In [30]:
deployment_details = best_pattern.inference_service.deploy(
    name="AutoAI RAG deployment - ibm_watsonx_ai documentataion",
    space_id=SPACE_ID,
    deploy_params={"tags": ["wx-autoai-rag"]},
)



######################################################################################

Synchronous deployment creation for id: '6d10fba7-1421-42de-9929-d2b033fa72d6' started

######################################################################################


initializing
Note: online_url and serving_urls are deprecated and will be removed in a future release. Use inference instead.
......
ready


-----------------------------------------------------------------------------------------------
Successfully finished deployment creation, deployment_id='c581e15d-ae4f-4f09-af92-c4d728ed5109'
-----------------------------------------------------------------------------------------------




### Test the deployed function

The RAG service is now deployed in the space. To test the solution, run the cell below. Questions have to be provided in the payload. Their format is provided below.

In [31]:
deployment_id = client.deployments.get_id(deployment_details)

payload = {"messages": [{"role": "user", "content": question}]}
score_response = client.deployments.run_ai_service(deployment_id, payload)
score_response

{'choices': [{'index': 0,
   'message': {'content': "Based on the provided context, the following industry players are mentioned as IBM's strategic partners:\n\n- Meta (AI@Meta)\n- Cohere (Cohere Command r+)\n- Databricks (Databricks introducing dbrx)\n- Mistral AI (Mistral 7B)\n- OpenAssistant (OpenAssistant conversations)\n- IBM Research (IBM Research leaders mentioned in acknowledgments)\n\nThe context does not explicitly list IBM's strategic partners, but these are some of the companies and organizations that are mentioned in relation to IBM in the given text.",
    'role': 'system'},
   'reference_documents': [{'metadata': {'document_id': 'granite_code_models_paper.md',
      'sequence_number': [57, 58, 59, 60, 61, 62, 63, 64, 65]},
     'page_content': 'We also compare Granite-8B-Code with CodeLlama-7B in Figure 5 and find that Granite-8B-Code-Instruct beats CodeLlama-7B-Instruct by 22%, 14% and 12% on AST Summary, Execution Summary and Overall accuracy respectively. Additionally

In [32]:
print(score_response["choices"][0]["message"]["content"])

Based on the provided context, the following industry players are mentioned as IBM's strategic partners:

- Meta (AI@Meta)
- Cohere (Cohere Command r+)
- Databricks (Databricks introducing dbrx)
- Mistral AI (Mistral 7B)
- OpenAssistant (OpenAssistant conversations)
- IBM Research (IBM Research leaders mentioned in acknowledgments)

The context does not explicitly list IBM's strategic partners, but these are some of the companies and organizations that are mentioned in relation to IBM in the given text.


<a id="Summary-and-next-steps"></a>
## Summary and next steps

You successfully completed this notebook!

You learned how to use AutoAI RAG with documents processed by the TextExtraction service.
 
Check out our _<a href="https://ibm.github.io/watsonx-ai-python-sdk/samples.html" target="_blank" rel="noopener no referrer">Online Documentation</a>_ for more samples, tutorials, documentation, how-tos, and blog posts. 

### Author:
 **Paweł Kocur**, Software Engineer at watsonx.ai.

Copyright © 2025-2026 IBM. This notebook and its source code are released under the terms of the MIT License.